# 04 -- Audio Spectrogram Transformer (AST)

Every model so far (baseline CNN, ResNet, EfficientNet) is a
convolutional network: local receptive fields that only see the whole
spectrogram after many stacked layers. AST (Gong, Chung & Glass, 2021)
takes a completely different approach, borrowed from Vision Transformer
(ViT): chop the spectrogram into a grid of small patches, embed each
patch as a token, and run standard transformer self-attention across
*all* patches at *every* layer. From layer one, a patch at time=0.1s,
2kHz can directly attend to a patch at time=2.9s, 500Hz -- no locality
bias at all, in exchange for needing more data (or, as here, pretraining)
to learn what structure to attend to.

We use `MIT/ast-finetuned-audioset-10-10-0.4593` -- AST-base (~86M
parameters) already fine-tuned on AudioSet (2M+ YouTube clips, 527 sound
event classes, includes some animal sounds but essentially no underwater
bioacoustic recordings). That pretraining is a much closer domain match
than ImageNet natural images for *general* audio structure, which makes
this an interesting three-way comparison in notebook 05: CNN-from-scratch
vs. CNN-from-ImageNet vs. transformer-from-AudioSet.

In [ ]:
import os
import subprocess
import sys
import importlib.util

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    # 1. Mount Drive and locate the project. Upload src/, configs/,
    #    pyproject.toml, requirements.txt to this path in My Drive first --
    #    NOT the multi-GB Watkins/ or results/ folders, those are handled
    #    separately below.
    from google.colab import drive
    drive.mount("/content/drive")

    PROJECT_DRIVE_PATH = "/content/drive/MyDrive/MarineMammals"  # <-- edit if you used a different path
    if not os.path.exists(f"{PROJECT_DRIVE_PATH}/src/watkins"):
        raise FileNotFoundError(
            f"Expected the project's src/ folder at {PROJECT_DRIVE_PATH}/src on Google Drive.\n"
            "Upload src/, configs/, pyproject.toml, and requirements.txt there "
            "(skip the multi-GB Watkins/ and results/ folders), or edit "
            "PROJECT_DRIVE_PATH above to match where you put them."
        )
    sys.path.insert(0, f"{PROJECT_DRIVE_PATH}/src")

    # 2. Install whatever Colab's base image doesn't already have. Deliberately
    #    does NOT touch torch/torchaudio/torchvision -- Colab's preinstalled
    #    versions are already matched to its GPU + CUDA build, and reinstalling
    #    this project's CPU-only wheels here would silently disable the GPU.
    needed = ["transformers", "timm", "soundfile", "datasets", "huggingface_hub", "pyarrow"]
    missing = [pkg for pkg in needed if importlib.util.find_spec(pkg) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

    # 3. Data goes on fast local/ephemeral disk (re-derivable from the public
    #    Hugging Face source, no reason to pay Drive's slow random-I/O tax on
    #    thousands of per-epoch file reads); results go on Drive so trained
    #    checkpoints/metrics survive a runtime disconnect.
    os.environ["WATKINS_DATA_ROOT"] = "/content/watkins_data"
    os.environ["WATKINS_RESULTS_ROOT"] = f"{PROJECT_DRIVE_PATH}/results"

    from watkins.data import DATA_ROOT
    if not (DATA_ROOT / "manifest.csv").exists():
        print("Materializing the Watkins dataset locally -- one-time per Colab runtime, ~10-15 min...")
        subprocess.run([sys.executable, "-m", "watkins.prepare_data"], check=True)
else:
    sys.path.insert(0, "../src")

from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt

from watkins.train import train_run, load_config
from watkins.evaluate import evaluate_checkpoint
from watkins.utils import count_parameters, get_device
from watkins.models.ast_model import build_ast_classifier

# Colab runs every notebook with cwd=/content regardless of where the
# .ipynb itself lives, so bare "../configs/..." paths silently miss.
# Resolve against the project root instead, which differs by environment.
PROJECT_ROOT = Path(PROJECT_DRIVE_PATH) if IN_COLAB else Path("..")


## 1. Why the CPU default is linear probing, not full fine-tuning

The development machine has no usable GPU (see the README for why: the
installed Quadro P520 is a Pascal-generation card current PyTorch CUDA
builds have dropped support for). AST-base is ~86M parameters --
fine-tuning all of them on CPU, one gradient step at a time, is slow.
`configs/ast_linear_probe.yaml` instead **freezes the pretrained
transformer entirely** and trains only a small classifier head (a
LayerNorm + Linear layer, sized for 54 species) on top of its pooled
768-dim output.

Because the backbone is frozen and never updates,
`watkins.models.ast_model.extract_embeddings` runs it over the dataset
**exactly once**, caching the resulting 768-dim vectors -- turning "train
a transformer" into "train logistic regression on fixed features," which
is fast even on CPU.

That default is driven by hardware, but the probe is not a consolation
prize. Run at full scale (`configs/gpu/ast_linear_probe.yaml`), it
reaches **0.507 test accuracy / 0.308 macro-F1** -- the second-best
accuracy of the five models in this project, for the cost of a single
forward pass over the dataset and a head that trains in seconds. That is
AudioSet pretraining transferring to underwater bioacoustics with *no*
adaptation of the backbone at all. Section 3 covers what unfreezing it
buys on top of that.

Check the trainable parameter count:

In [ ]:
ast_frozen = build_ast_classifier(num_classes=54, freeze_backbone=True)
print(f"AST total params:      {count_parameters(ast_frozen):,}")
print(f"AST trainable params:  {count_parameters(ast_frozen, trainable_only=True):,}")
print(f"device in use:         {get_device()}")


## 2. Train the linear probe

The slow part is the one-time embedding extraction (a full forward pass
through 86M frozen parameters for every training/val/test clip); the
"training" loop afterwards is nearly instant. Expect the cell below to
take noticeably longer to *start* producing epoch logs than the CNN
notebooks did -- that's the embedding extraction, not the training loop,
so don't mistake it for a hang.

In [ ]:
cfg = load_config(PROJECT_ROOT / "configs/ast_linear_probe.yaml")
# The YAML's run_name is "ast_linear_probe", which notebook 05 lists alongside
# the full configs/gpu/ runs. Rename so this 30%-subset demo doesn't land in
# that comparison table as if it were a full-scale result.
cfg["run_name"] = "ast_linear_probe_demo"
cfg["subset_frac"] = 0.3  # trim for a faster in-notebook demo; drop for the full run

ast_result = train_run(cfg)
print(f"AST linear probe: test_acc={ast_result['test_acc']:.3f}  test_f1={ast_result['test_f1']:.3f}")


In [ ]:
evaluate_checkpoint(ast_result["checkpoint_path"])


## 3. Full fine-tuning: what unfreezing the backbone actually bought

`configs/ast_finetune.yaml` unfreezes the whole backbone
(`freeze_backbone: false`) and trains end-to-end with a tiny learning
rate (1e-5 -- large learning rates destroy pretrained weights fast) and a
small batch size. That CPU-sized config defaults to a 25% data subset and
3 epochs specifically so it's *runnable* as a demo on this machine, not
because that's a serious training budget.

The real run is `configs/gpu/ast_finetune.yaml` on a rented GPU, and it
has already been done -- it's the headline result in the README:

| | test accuracy | test macro-F1 |
|---|---|---|
| `ast_linear_probe` (frozen) | 0.507 | 0.308 |
| `ast_finetune` (all 86M unfrozen) | **0.576** | **0.358** |

So backprop through all 86M parameters is worth **+0.069 accuracy and
+0.050 macro-F1** over the probe -- a real margin, and the direction you'd
predict given the domain gap between AudioSet's general sound-event
classes and this dataset's marine-mammal species. Cost: roughly one hour
on an RTX 4090 at bf16 over the full 15k clips (all five GPU configs
together came to ~2 hours and well under $1), versus seconds for the
probe once its embeddings are cached. See `docs/runpod_run.md`.

Two things worth noticing before you read that as a clean win:

1. **It early-stopped after 9 epochs**, with train macro-F1 above 0.95
   while validation plateaued near 0.35 -- 86M parameters against ~9,400
   training clips overfits fast. See `results/logs/ast_finetune_gpu.csv`.
2. **On macro-F1 it only ties EfficientNet-B0** (0.358 vs 0.357), a model
   with 1/21 the parameters. The accuracy win is real, but macro-F1 says
   the extra capacity bought performance on the well-represented species
   rather than on the long tail -- which is the part of this problem that
   is actually hard. Notebook 05 makes that comparison directly.

If you want to watch the fine-tune run on CPU anyway, at a scale that
finishes rather than one that produces a meaningful number, uncomment
below:

In [ ]:
# finetune_cfg = load_config(PROJECT_ROOT / "configs/ast_finetune.yaml")
# finetune_cfg["subset_frac"] = 0.1   # shrink further for a CPU demo
# finetune_cfg["epochs"] = 1
# finetune_result = train_run(finetune_cfg)
# print(finetune_result["test_acc"], finetune_result["test_f1"])


## 4. Note on input length

The pretrained checkpoint expects up to 1024 time frames (~10.24s of
audio); our clips are cropped/padded to 3.0s (~300 frames after AST's own
feature extraction), so roughly 70% of every input is zero-padding. That
wastes part of the model's positional embedding range and is worth
remembering when interpreting AST's results relative to the CNNs, which
see the full spectrogram with no padding. See `docs/next_steps.md` for
the exercise of trimming `max_length` and interpolating positional
embeddings for shorter clips -- a legitimate way AST could likely be made
to perform better here than the default config lets it.

## Exercises

1. Compare the linear probe's per-class F1 (from `evaluate_checkpoint`
   above) against the frozen-ResNet18 numbers from notebook 03. Both are
   "frozen pretrained backbone + linear head" -- does the AudioSet
   pretraining (closer domain, general audio) beat ImageNet pretraining
   (further domain, natural images), as you'd predict?
2. `ast_linear_probe.yaml` uses `lr: 0.01` -- much higher than the CNN
   configs. Why is a high learning rate safe here but would be
   destructive for fine-tuning the full backbone?
3. Both full-scale AST runs already ship in `results/metrics/`
   (`ast_linear_probe_gpu_eval.json` and `ast_finetune_gpu_eval.json`).
   Load both and work out *where* the fine-tune's +0.069 accuracy came
   from: compare per-class F1 on the five best-represented species
   against the twenty worst-represented ones. Is the gain spread evenly,
   or concentrated in the head of the distribution? (Section 3 states the
   conclusion -- confirm it from the numbers yourself.)
4. Try `extract_embeddings` yourself on just the *test* set and use
   `sklearn.decomposition.PCA` or `sklearn.manifold.TSNE` to visualize
   the 768-dim embeddings in 2D, colored by species. Do any of the
   54 species separate visually before any classifier head is even
   trained on them? Given the severe class imbalance, focus on the
   best-represented handful first (killer whale, sperm whale,
   long-finned pilot whale, ...).
5. The fine-tune early-stopped after 9 epochs with train macro-F1 above
   0.95 and validation stuck near 0.35 -- textbook overfitting on ~9,400
   training clips. `docs/next_steps.md` lists the untried levers, cheapest
   first: turn on `spec_augment` (the GPU config has it off), raise
   `weight_decay`, add LR warmup + cosine decay, or freeze the lower
   transformer blocks so only the upper layers adapt. Which would you try
   first, and why? Each is worth ~15 minutes of rented GPU time to test.